# Data ingestion
For the Databricks project, DLT part, we want to process stock market data. 

To do this, I chose different companies covering several sectors: NVIDIA, Lockheed Martin, Google (Alphabet), and TotalEnergies.

In [0]:
import requests, time
from pyspark.sql import functions as F

# Normally we should stock the API_KEY in the Databricks Secrets or Azure Key Vault, and reference it here.
API_KEY = 'AEHFJB7KWSYUD2OU'
symbols = ['NVDA', 'LMT', 'GOOGL', 'TTE']

### Daily stock data

In [0]:
stock_rows = []
meta_rows = []

for symbol in symbols:
    url = f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&apikey={API_KEY}"
    # We only have 5 requests per minute so we put a timer
    time.sleep(15)

    r = requests.get(url)
    data = r.json()
    
    # Check if we don't have any data. Possible to put a log instead
    if "Time Series (Daily)" not in data:
        print(f"API error for {symbol}")
        continue
    
    # Get separated data
    time_series = data["Time Series (Daily)"]
    meta_data = data["Meta Data"]

    # Get meta data (for each symbol)
    meta_rows.append((
        meta_data["2. Symbol"],
        meta_data["1. Information"],
        meta_data["3. Last Refreshed"],
        meta_data["4. Output Size"],
        meta_data["5. Time Zone"]
    ))

    # Get stock data for each date (for each symbol)
    for date, values in time_series.items():
        stock_rows.append((
            symbol,
            date,
            float(values['1. open']),
            float(values['2. high']),
            float(values['3. low']),
            float(values['4. close']),
            int(values['5. volume'])
        ))

# Determine "schema" for stock symbol
columns = ["symbol", "date", "open", "high", "low", "close", "volume"]

# Transform the rows into a DF and add ingestion time
stock_df = ( 
    spark
    .createDataFrame(stock_rows, columns)
    .withColumn("ingestion_time", F.current_timestamp())
)

# Determine "schema" for Meta data
meta_columns = [
    "symbol",
    "information",
    "last_refreshed",
    "output_size",
    "time_zone"
]

# Transform the rows into a DF and add ingestion time
meta_df = (
    spark
    .createDataFrame(meta_rows, meta_columns)
    .withColumn("ingestion_time", F.current_timestamp())
)

display(stock_df)
display(meta_df)


symbol,date,open,high,low,close,volume,ingestion_time
NVDA,2026-02-17,181.75,187.15,179.18,184.97,162276860,2026-02-18T14:40:19.676Z
NVDA,2026-02-13,187.475,187.5,181.59,182.81,161888021,2026-02-18T14:40:19.676Z
NVDA,2026-02-12,193.03,193.61,186.51,186.94,189932491,2026-02-18T14:40:19.676Z
NVDA,2026-02-11,192.45,193.26,188.77,190.05,144192685,2026-02-18T14:40:19.676Z
NVDA,2026-02-10,191.38,192.48,188.12,188.54,136764825,2026-02-18T14:40:19.676Z
NVDA,2026-02-09,184.26,193.66,183.95,190.04,196387351,2026-02-18T14:40:19.676Z
NVDA,2026-02-06,176.69,187.0,174.6,185.41,231346241,2026-02-18T14:40:19.676Z
NVDA,2026-02-05,174.925,176.815,171.03,171.88,206312890,2026-02-18T14:40:19.676Z
NVDA,2026-02-04,179.46,179.58,171.91,174.19,207014116,2026-02-18T14:40:19.676Z
NVDA,2026-02-03,186.24,186.27,176.23,180.34,202006430,2026-02-18T14:40:19.676Z


symbol,information,last_refreshed,output_size,time_zone,ingestion_time
NVDA,"Daily Prices (open, high, low, close) and Volumes",2026-02-17,Compact,US/Eastern,2026-02-18T14:40:20.202Z
LMT,"Daily Prices (open, high, low, close) and Volumes",2026-02-17,Compact,US/Eastern,2026-02-18T14:40:20.202Z
GOOGL,"Daily Prices (open, high, low, close) and Volumes",2026-02-17,Compact,US/Eastern,2026-02-18T14:40:20.202Z
TTE,"Daily Prices (open, high, low, close) and Volumes",2026-02-17,Compact,US/Eastern,2026-02-18T14:40:20.202Z


In [0]:
stock_df.write.mode("append").saveAsTable("jrvs_dlt.01_bronze.stock_data")
meta_df.write.mode("append").saveAsTable("jrvs_dlt.01_bronze.stock_meta_data")

### Company info

In [0]:
company_rows = []

for symbol in symbols:
    url_company = f'https://www.alphavantage.co/query?function=OVERVIEW&symbol={symbol}&apikey={API_KEY}'
    time.sleep(15)

    company_r = requests.get(url_company)
    company_data = company_r.json()

    if not company_data or "Symbol" not in company_data:
        print(f"API error for {symbol}")
        continue

    company_rows.append(company_data)

company_df = spark.createDataFrame(company_rows) \
    .withColumn("ingestion_time", F.current_timestamp())

display(company_df)


200DayMovingAverage,50DayMovingAverage,52WeekHigh,52WeekLow,Address,AnalystRatingBuy,AnalystRatingHold,AnalystRatingSell,AnalystRatingStrongBuy,AnalystRatingStrongSell,AnalystTargetPrice,AssetType,Beta,BookValue,CIK,Country,Currency,Description,DilutedEPSTTM,DividendDate,DividendPerShare,DividendYield,EBITDA,EPS,EVToEBITDA,EVToRevenue,ExDividendDate,Exchange,FiscalYearEnd,ForwardPE,GrossProfitTTM,Industry,LatestQuarter,MarketCapitalization,Name,OfficialSite,OperatingMarginTTM,PEGRatio,PERatio,PercentInsiders,PercentInstitutions,PriceToBookRatio,PriceToSalesRatioTTM,ProfitMargin,QuarterlyEarningsGrowthYOY,QuarterlyRevenueGrowthYOY,ReturnOnAssetsTTM,ReturnOnEquityTTM,RevenuePerShareTTM,RevenueTTM,Sector,SharesFloat,SharesOutstanding,Symbol,TrailingPE,ingestion_time
171.68,184.41,212.18,86.6,"2788 SAN TOMAS EXPRESSWAY, SANTA CLARA, CA, UNITED STATES, 95051",48,3,1,12,0,253.88,Common Stock,2.314,4.892,1045810,USA,USD,"Nvidia Corporation is an American multinational technology company incorporated in Delaware and based in Santa Clara, California. It designs graphics processing units (GPUs) for the gaming and professional markets, as well as system on a chip units (SoCs) for the mobile computing and automotive market.",4.09,2025-12-26,0.04,0.0002,112696001000,4.09,37.33,23.76,2025-12-04,NASDAQ,January,24.04,131092996000,SEMICONDUCTORS,2025-10-31,4503464575000,NVIDIA Corporation,https://www.nvidia.com,0.632,0.714,45.22,4.329,69.840,37.81,24.06,0.53,0.667,0.625,0.535,1.074,7.67,187141997000,TECHNOLOGY,23331402000,24305000000,NVDA,45.22,2026-02-18T14:41:39.146Z
485.27,541.48,656.34,404.05,"6801 ROCKLEDGE DRIVE, BETHESDA, MD, UNITED STATES, 20817",5,14,1,1,0,657.58,Common Stock,0.23,29.35,936468,USA,USD,"Lockheed Martin Corporation is an American aerospace, defense, information security, and technology company with worldwide interests. It is headquartered in North Bethesda, Maryland, in the Washington, D.C., area.",21.47,2026-03-27,13.35,0.0205,8285000000,21.47,19.22,2.235,2026-03-02,NYSE,December,21.74,7685000000,AEROSPACE & DEFENSE,2025-12-31,150311387000,Lockheed Martin Corporation,https://www.lockheedmartin.com,0.0901,1.393,30.26,0.085,75.358,22.34,2.003,0.0669,1.61,0.091,0.0757,0.769,322.51,75048002000,INDUSTRIALS,196042000,230080000,LMT,30.26,2026-02-18T14:41:39.146Z
242.85,321.22,349,140.14,"1600 AMPHITHEATRE PARKWAY, MOUNTAIN VIEW, CA, UNITED STATES, 94043",47,8,0,12,0,373.24,Common Stock,1.086,34.35,1652044,USA,USD,"Alphabet Inc. is an American multinational conglomerate headquartered in Mountain View, California. It was created through a restructuring of Google on October 2, 2015, and became the parent company of Google and several former Google subsidiaries. The two co-founders of Google remained as controlling shareholders, board members, and employees at Alphabet. Alphabet is the world's fourth-largest technology company by revenue and one of the world's most valuable companies.",10.81,2026-03-16,0.83,0.0027,150175007000,10.81,20.1,9.02,2026-03-09,NASDAQ,December,26.95,240300999000,INTERNET CONTENT & INFORMATION,2025-12-31,3653536055000,Alphabet Inc Class A,https://abc.xyz,0.316,2.318,27.94,0.569,80.939,8.91,9.07,0.328,0.311,0.18,0.154,0.357,33.25,402835997000,COMMUNICATION SERVICES,10833226000,5822000000,GOOGL,27.94,2026-02-18T14:41:39.146Z
62.64,67.26,77.77,50.39,"TOUR COUPOLE - 2, COURBEVOIE, FRANCE, 92078",2,5,0,1,0,71.94,Common Stock,0.253,53.21,879764,USA,USD,"TotalEnergies SE is a global integrated oil and gas company. The company is headquartered in Paris, France.",6.19,2025-10-22,3.682,0.0524,35641000000,6.19,4.944,1.07,2026-03-31,NYSE,December,11.51,65988002000,OIL & GAS INTEGRATED,2025-09-30,159895323000,TotalEnergies SE ADR,https://www.totalenergies.com,0.125,1.856,12.06,5.275,46.911,1.432,0.899,0.0772,0.708,-0.076,0.0486,0.122,82.41,183533994000,ENERGY,1760116000,2142507000,TTE,12.06,2026-02-18T14:41:39.146Z


In [0]:
company_df.write.mode("append").saveAsTable("jrvs_dlt.01_bronze.company_data")